In [ ]:
# IMPORT LIBRARIES AND SET PROJECT PATHS

import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer

In [20]:
# SET PATHS

BASE_DIR = r"C:\Users\acer\Documents\Clinical_Trial_Classification"
INPUT_PATH = os.path.join(
    BASE_DIR, "outputs", "preprocessed_clinical_trials.csv"
)
OUTPUT_DIR = os.path.join(BASE_DIR, "outputs")
MODEL_DIR = os.path.join(BASE_DIR, "models")

print("Input file exists:", os.path.exists(INPUT_PATH))
print("Feature engineering setup completed.")

Input file exists: True
Feature engineering setup completed.


In [ ]:
# LOAD PREPROCESSED DATASET

from IPython.display import display as show_table

df = pd.read_csv(INPUT_PATH)

print("Dataset loaded successfully.")
print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())
print("\nFirst 5 rows:")
show_table(df.head())

Dataset loaded successfully.
Dataset shape: (60337, 18)

Column names:
['source_condition_query', 'nct_id', 'title', 'official_title', 'brief_summary', 'conditions', 'interventions', 'overall_status', 'study_type', 'phase', 'sex', 'minimum_age', 'maximum_age', 'healthy_volunteers', 'eligibility_criteria', 'clinicaltrials_url', 'cleaned_summary', 'processed_summary']

First 5 rows:


,source_condition_query,nct_id,title,official_title,brief_summary,conditions,interventions,overall_status,study_type,phase,sex,minimum_age,maximum_age,healthy_volunteers,eligibility_criteria,clinicaltrials_url,cleaned_summary,processed_summary
0,breast cancer,NCT03676114,Effect of Perioperative Low Dose Ketamine on P...,Effect of Perioperative Low Dose Ketamine on P...,Breast cancer patients often have perioperativ...,Breast Cancer,ketamine | Normal saline,UNKNOWN,INTERVENTIONAL,PHASE4,FEMALE,20 Years,65 Years,False,Inclusion Criteria:\n\n1. American Society of ...,https://clinicaltrials.gov/study/NCT03676114,breast cancer patients often have perioperativ...,breast cancer patient often perioperative emot...
1,breast cancer,NCT02941614,Implementing Systematic Distress Screening in ...,Implementing Systematic Distress Screening in ...,Many breast cancer patients experience psychol...,Breast Cancer,Distress screening,COMPLETED,OBSERVATIONAL,NaN,ALL,18 Years,NaN,False,Inclusion Criteria:\n\n* Newly diagnosed with ...,https://clinicaltrials.gov/study/NCT02941614,many breast cancer patients experience psychol...,many breast cancer patient experience psycholo...
2,breast cancer,NCT04509063,Investigating Public Enthusiasm for Mammograph...,Investigating Public Enthusiasm for Mammograph...,"Based on an American study by Scherer et al., ...",Breast Neoplasm Female | Mammography Screening...,Information about hypothetical mammography scr...,COMPLETED,INTERVENTIONAL,NaN,FEMALE,44 Years,49 Years,True,Inclusion Criteria:\n\n* Residence: Central De...,https://clinicaltrials.gov/study/NCT04509063,based on an american study by scherer et al it...,base american study scherer et al hypothesize ...
3,breast cancer,NCT04327063,MIRs 04 : Interpectoral Nerve Block With Ropiv...,A Double-blind Randomized Trial of Interpector...,Compare the effect of ropivacaine versus place...,Malignant Neoplasm of Breast,Saline | Ropivacaine,COMPLETED,INTERVENTIONAL,PHASE3,FEMALE,18 Years,85 Years,False,Inclusion Criteria:\n\n1. Women with non-metas...,https://clinicaltrials.gov/study/NCT04327063,compare the effect of ropivacaine versus place...,compare effect ropivacaine versus placebo pect...
4,breast cancer,NCT06778863,A Study of CLSP-1025 in Adult Patients With So...,GUARDIAN-101: A Phase 1 Dose Escalation and Ex...,Phase 1 dose escalation and expansion study of...,Advanced Solid Tumor | Unresectable Solid Tumo...,CLSP-1025,RECRUITING,INTERVENTIONAL,PHASE1,ALL,18 Years,NaN,False,Key Inclusion Criteria:\n\n* Patients must be ...,https://clinicaltrials.gov/study/NCT06778863,phase dose escalation and expansion study of c...,phase dose escalation expansion study clsp fir...


In [ ]:
# CHECK TARGET CATEGORY DISTRIBUTION

target_column = "source_condition_query"

print("Number of disease categories:", df[target_column].nunique())
print("\nDisease category distribution:")
print(df[target_column].value_counts())

print("\nMissing target values:", df[target_column].isna().sum())

Number of disease categories: 8

Disease category distribution:
source_condition_query
breast cancer                            16301
type 2 diabetes                          11467
covid-19                                 10153
anxiety                                   9286
chronic obstructive pulmonary disease     6181
rheumatoid arthritis                      3637
glaucoma                                  2173
sickle cell anemia                        1139
Name: count, dtype: int64

Missing target values: 0


In [ ]:
# ENCODE TARGET CATEGORIES

label_encoder = LabelEncoder()

df["target"] = label_encoder.fit_transform(
    df["source_condition_query"]
)

print("Target encoding completed.")
print("\nDisease category mapping:")

for category, encoded_value in zip(
    label_encoder.classes_,
    label_encoder.transform(label_encoder.classes_)
):
    print(f"{category}: {encoded_value}")

print("\nEncoded target shape:", df["target"].shape)

Target encoding completed.

Disease category mapping:
anxiety: 0
breast cancer: 1
chronic obstructive pulmonary disease: 2
covid-19: 3
glaucoma: 4
rheumatoid arthritis: 5
sickle cell anemia: 6
type 2 diabetes: 7

Encoded target shape: (60337,)


In [ ]:
# SEPARATE FEATURES AND TARGET

X = df["processed_summary"]
y = df["target"]

print("Text feature shape:", X.shape)
print("Target shape:", y.shape)
print("Missing text values:", X.isna().sum())
print("Missing target values:", y.isna().sum())

Text feature shape: (60337,)
Target shape: (60337,)
Missing text values: 0
Missing target values: 0


In [ ]:
# SPLIT TRAINING AND TESTING DATA

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training text samples:", X_train.shape[0])
print("Testing text samples:", X_test.shape[0])
print("Training target samples:", y_train.shape[0])
print("Testing target samples:", y_test.shape[0])

Training text samples: 48269
Testing text samples: 12068
Training target samples: 48269
Testing target samples: 12068


In [ ]:
# TF-IDF FEATURE EXTRACTION

tfidf_vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

print("TF-IDF feature extraction completed.")
print("Training feature matrix:", X_train_tfidf.shape)
print("Testing feature matrix:", X_test_tfidf.shape)
print("Number of extracted features:", len(tfidf_vectorizer.get_feature_names_out()))

TF-IDF feature extraction completed.
Training feature matrix: (48269, 10000)
Testing feature matrix: (12068, 10000)
Number of extracted features: 10000


In [28]:
# SAVE ENGINEERED FEATURES

from scipy.sparse import save_npz

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

save_npz(
    os.path.join(OUTPUT_DIR, "X_train_tfidf.npz"),
    X_train_tfidf
)
save_npz(
    os.path.join(OUTPUT_DIR, "X_test_tfidf.npz"),
    X_test_tfidf
)

pd.DataFrame({"target": y_train}).to_csv(
    os.path.join(OUTPUT_DIR, "y_train.csv"), index=False
)
pd.DataFrame({"target": y_test}).to_csv(
    os.path.join(OUTPUT_DIR, "y_test.csv"), index=False
)

import joblib
joblib.dump(
    tfidf_vectorizer,
    os.path.join(MODEL_DIR, "tfidf_vectorizer.pkl")
)
joblib.dump(
    label_encoder,
    os.path.join(MODEL_DIR, "label_encoder.pkl")
)

print("Engineered features and transformation objects saved successfully.")

Engineered features and transformation objects saved successfully.
